# 错题绑定测试台

这个 notebook 用来测试 `wrong_question_binder.py` 的通用性。

使用方式很简单：
- 先只填写 `题目`、`标准答案`、`参考解析`
- 默认不填写学生答案
- 运行后看中文说明输出：粗路由、主绑定、Top-K 候选、分数拆解、是否真的启用了 embedding

说明：
- 默认只往 `scratch/wrong_question_binder_playground/` 写测试结果
- 不会修改正式叶子库


In [16]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from wrong_question_binder import WrongQuestionBinder, summarize_result

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'wrong_question_binder_playground'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = SCRATCH_DIR / 'binder_test_output.json'
OUTPUT_NO_EMBED_JSON = SCRATCH_DIR / 'binder_test_output_no_embedding.json'


## 1. 在这里填写题目

这里只需要先填三块：
- `题目`
- `标准答案`
- `参考解析`

如果暂时没有学生答案，就保持空字符串。

In [17]:
题目 = '''
已知点A（2，1）在双曲线𝐶：𝑥^2/𝑎^2−𝑦^2/𝑎^2−1=1（𝑎＞1）上，直线l交C于P，Q两点，直线AP，AQ的斜率之和为0．求l的斜率
'''.strip()

标准答案 = '''
求双曲线 C 的方程及其渐近线 因为点 A(2,1) 在双曲线 x^2/a^2 - y^2/(a^2-1) = 1  (a>1) 上，
所以 4/a^2 - 1/(a^2-1) = 1. 两边同乘 a^2(a^2-1)，得  4(a^2-1) - a^2 = a^2(a^2-1), 整理得 a^4 - 4a^2 + 4 = 0,
即 (a^2 - 2)^2 = 0.  所以  a^2 = 2.  故双曲线 C 的方程为  x^2/2 - y^2 = 1，  即  x^2 - 2y^2 = 2.
由  x^2 - 2y^2 = 0  得双曲线的渐近线方程为  y = ±(√2/2)x.  （2）求直线 l 的斜率
设直线 l 的方程为  y = kx + m，设  P(x1,y1), Q(x2,y2).由  y = kx + m  与  x^2 - 2y^2 = 2  联立，得  x^2 - 2(kx+m)^2 = 2,
整理得  (1-2k^2)x^2 - 4mkx - 2m^2 - 2 = 0.  因此 x1, x2 是方程  (1-2k^2)x^2 - 4mkx - 2m^2 - 2 = 0  的两个实根，
由韦达定理得  x1 + x2 = -4mk/(2k^2-1),  x1x2 = 2(m^2+1)/(2k^2-1).  又由题意，直线 AP，AQ 的斜率之和为 0，即  (y1-1)/(x1-2) + (y2-1)/(x2-2) = 0.  将  y1 = kx1 + m, y2 = kx2 + m
代入，整理得  2kx1x2 + (m-1-2k)(x1+x2) - 4(m-1) = 0.
再将韦达定理所得结果代入，化简得
(k+1)(m+2k-1) = 0. 于是有  k = -1  或 m = 1 - 2k. 当  m = 1 - 2k  时，直线 l 的方程为  y = kx + 1 - 2k，  即  y - 1 = k(x - 2),  说明直线 l 经过点 A(2,1)，这与题意不符。 因此只能有  k = -1.  所以直线 l 的斜率为  -1.
'''.strip()

参考解析 = '''
由点 A(2,1) 在双曲线 ( \frac{x^2}{a^2}-\frac{y^2}{a^2-1}=1\ (a>1) ) 上，代入得 ( \frac{4}
  {a^2}-\frac{1}{a^2-1}=1 )，整理可得 ( a^4-4a^2+4=0 )，所以 ( a^2=2 )，故双曲线方程为 ( \frac{x^2}
  {2}-y^2=1 )，即 ( x^2-2y^2=2 )，其渐近线方程为 ( y=\pm \frac{\sqrt{2}}{2}x )。设直线 ( l ) 的方程
  为 ( y=kx+m )，交双曲线于 ( P(x_1,y_1),Q(x_2,y_2) )，联立可得 ( (1-2k^2)x^2-4mkx-2m^2-2=0 )，于是
  由韦达定理可得 ( x_1+x_2=-\frac{4mk}{2k^2-1} )，( x_1x_2=\frac{2m^2+2}{2k^2-1} )。又因为直线
  ( AP )、( AQ ) 的斜率之和为 0，所以 ( \frac{y_1-1}{x_1-2}+\frac{y_2-1}{x_2-2}=0 )，将
  ( y_1=kx_1+m,\ y_2=kx_2+m ) 代入并结合韦达定理化简，得到 ( (k+1)(m+2k-1)=0 )。若 ( m=1-2k )，则直
  线方程可写成 ( y-1=k(x-2) )，说明直线 ( l ) 过点 A(2,1)，与题意不符，因此只能有 ( k=-1 )。所以双
  曲线 ( C ) 的方程为 ( \frac{x^2}{2}-y^2=1 )，渐近线方程为 ( y=\pm \frac{\sqrt{2}}{2}x )，直线
  ( l ) 的斜率为 (-1)。
'''.strip()

学生答案 = ''
教师备注 = ''

题型 = '证明题'
来源名称 = ''
来源章节 = ''
标签 = []

print('当前题目已载入。')
print('题目：', 题目)
print('标准答案：', 标准答案)
print('参考解析：', 参考解析[:120] + ('...' if len(参考解析) > 120 else ''))

当前题目已载入。
题目： 已知点A（2，1）在双曲线𝐶：𝑥^2/𝑎^2−𝑦^2/𝑎^2−1=1（𝑎＞1）上，直线l交C于P，Q两点，直线AP，AQ的斜率之和为0．求l的斜率
标准答案： 求双曲线 C 的方程及其渐近线 因为点 A(2,1) 在双曲线 x^2/a^2 - y^2/(a^2-1) = 1  (a>1) 上，
所以 4/a^2 - 1/(a^2-1) = 1. 两边同乘 a^2(a^2-1)，得  4(a^2-1) - a^2 = a^2(a^2-1), 整理得 a^4 - 4a^2 + 4 = 0,
即 (a^2 - 2)^2 = 0.  所以  a^2 = 2.  故双曲线 C 的方程为  x^2/2 - y^2 = 1，  即  x^2 - 2y^2 = 2.
由  x^2 - 2y^2 = 0  得双曲线的渐近线方程为  y = ±(√2/2)x.  （2）求直线 l 的斜率
设直线 l 的方程为  y = kx + m，设  P(x1,y1), Q(x2,y2).由  y = kx + m  与  x^2 - 2y^2 = 2  联立，得  x^2 - 2(kx+m)^2 = 2,
整理得  (1-2k^2)x^2 - 4mkx - 2m^2 - 2 = 0.  因此 x1, x2 是方程  (1-2k^2)x^2 - 4mkx - 2m^2 - 2 = 0  的两个实根，
由韦达定理得  x1 + x2 = -4mk/(2k^2-1),  x1x2 = 2(m^2+1)/(2k^2-1).  又由题意，直线 AP，AQ 的斜率之和为 0，即  (y1-1)/(x1-2) + (y2-1)/(x2-2) = 0.  将  y1 = kx1 + m, y2 = kx2 + m
代入，整理得  2kx1x2 + (m-1-2k)(x1+x2) - 4(m-1) = 0.
再将韦达定理所得结果代入，化简得
(k+1)(m+2k-1) = 0. 于是有  k = -1  或 m = 1 - 2k. 当  m = 1 - 2k  时，直线 l 的方程为  y = kx + 1 - 2k，  即  y - 1 = k(x - 2),  说明直线 l 经过点 A(2,1)，这与题意不符。 因此只能有  k = -1.  所以直线 l 的斜率为  -1.
参考解

## 2. 测试参数

这里保留 `API_KEY` 变量，但默认留空。

需要测试 embedding 时，再临时把有效 key 粘到 `API_KEY` 这一行，跑完后建议清空。

如果你想看“不用 embedding”时的效果，可以把 `启用Embedding` 改成 `False`。

In [18]:
启用Embedding = True
使用默认凭证 = False
API_KEY = ''  # 需要测试 embedding 时，再临时填入有效 key

TOP_K = 5
MAX_SECONDARY = 2
COARSE_K = 3
GLOBAL_RECALL = 30
SPARSE_RECALL = 24
EMBEDDING_RECALL = 24

环境里有Key = bool(os.getenv('AZURE_AI_API_KEY') or os.getenv('AZURE_OPENAI_API_KEY'))
print({
    '启用Embedding': 启用Embedding,
    '使用默认凭证': 使用默认凭证,
    '环境里有Key': 环境里有Key,
    '输出文件': str(OUTPUT_JSON),
})

{'启用Embedding': True, '使用默认凭证': False, '环境里有Key': False, '输出文件': '/Users/xumuchi/Desktop/TeachAgent/scratch/wrong_question_binder_playground/binder_test_output.json'}


## 3. 组装输入记录

这一格会把你上面填的内容转成 binder 需要的标准格式。

In [19]:
QUESTION_RECORD = {
    'question_payload': {
        'stem': 题目,
        'question_type': 题型,
        'correct_answer': 标准答案,
        'solution_text': 参考解析,
    }
}

if 学生答案.strip():
    QUESTION_RECORD['question_payload']['student_answer'] = 学生答案.strip()
if 教师备注.strip():
    QUESTION_RECORD['question_payload']['teacher_comment'] = 教师备注.strip()
if 来源名称.strip():
    QUESTION_RECORD['question_payload']['source_name'] = 来源名称.strip()
if 来源章节.strip():
    QUESTION_RECORD['question_payload']['source_section'] = 来源章节.strip()
if 标签:
    QUESTION_RECORD['question_payload']['tags'] = 标签

print(json.dumps(QUESTION_RECORD, ensure_ascii=False, indent=2))

{
  "question_payload": {
    "stem": "已知点A（2，1）在双曲线𝐶：𝑥^2/𝑎^2−𝑦^2/𝑎^2−1=1（𝑎＞1）上，直线l交C于P，Q两点，直线AP，AQ的斜率之和为0．求l的斜率",
    "question_type": "证明题",
    "correct_answer": "求双曲线 C 的方程及其渐近线 因为点 A(2,1) 在双曲线 x^2/a^2 - y^2/(a^2-1) = 1  (a>1) 上，\n所以 4/a^2 - 1/(a^2-1) = 1. 两边同乘 a^2(a^2-1)，得  4(a^2-1) - a^2 = a^2(a^2-1), 整理得 a^4 - 4a^2 + 4 = 0,\n即 (a^2 - 2)^2 = 0.  所以  a^2 = 2.  故双曲线 C 的方程为  x^2/2 - y^2 = 1，  即  x^2 - 2y^2 = 2.\n由  x^2 - 2y^2 = 0  得双曲线的渐近线方程为  y = ±(√2/2)x.  （2）求直线 l 的斜率\n设直线 l 的方程为  y = kx + m，设  P(x1,y1), Q(x2,y2).由  y = kx + m  与  x^2 - 2y^2 = 2  联立，得  x^2 - 2(kx+m)^2 = 2,\n整理得  (1-2k^2)x^2 - 4mkx - 2m^2 - 2 = 0.  因此 x1, x2 是方程  (1-2k^2)x^2 - 4mkx - 2m^2 - 2 = 0  的两个实根，\n由韦达定理得  x1 + x2 = -4mk/(2k^2-1),  x1x2 = 2(m^2+1)/(2k^2-1).  又由题意，直线 AP，AQ 的斜率之和为 0，即  (y1-1)/(x1-2) + (y2-1)/(x2-2) = 0.  将  y1 = kx1 + m, y2 = kx2 + m\n代入，整理得  2kx1x2 + (m-1-2k)(x1+x2) - 4(m-1) = 0.\n再将韦达定理所得结果代入，化简得\n(k+1)(m+2k-1) = 0. 于是有  k = -1  或 m = 1 - 2k. 当  m = 1 - 2k  时，直线 l 的方程为  y = kx + 1 - 2k，  即 

## 4. 初始化 Binder

这里会告诉你两件事：
- 叶子 embedding 索引是否已经加载
- 本次运行是否真的具备调用 embedding 的能力

In [20]:
binder = WrongQuestionBinder.from_environment(
    enable_embeddings=启用Embedding,
    api_key=API_KEY,
    use_default_credential=使用默认凭证,
)

print({
    '已加载Embedding索引': binder.embedding_index is not None,
    '本次可调用Embedding': binder.embedding_client is not None,
    '启用Embedding开关': binder.enable_embeddings,
})

{'已加载Embedding索引': True, '本次可调用Embedding': True, '启用Embedding开关': True}


## 4.1 Embedding 自检

这一格专门回答一个问题：为什么这次可能没有真正调用 embedding。

In [21]:
def 中文Embedding自检(binder_obj, *, 启用Embedding, 使用默认凭证, API_KEY):
    环境里有Key = bool(os.getenv('AZURE_AI_API_KEY') or os.getenv('AZURE_OPENAI_API_KEY'))
    手动传入Key = bool(API_KEY)

    print('一、当前状态')
    print('启用Embedding开关：', 启用Embedding)
    print('已加载Embedding索引：', binder_obj.embedding_index is not None)
    print('本次可调用Embedding：', binder_obj.embedding_client is not None)
    print('环境里有Key：', 环境里有Key)
    print('手动传入API_KEY：', 手动传入Key)
    print('使用默认凭证：', 使用默认凭证)
    print()

    print('二、诊断结论')
    if not 启用Embedding:
        print('你主动关闭了 embedding，所以这次不会调用 embedding。')
    elif binder_obj.embedding_index is None:
        print('叶子 embedding 索引没有加载成功，所以这次不可能调用 embedding。')
    elif binder_obj.embedding_client is not None:
        print('本次已经具备调用 embedding 的能力。只要后续 bind 正常执行，就会真的发起 embedding 检索。')
    elif not (环境里有Key or 手动传入Key or 使用默认凭证):
        print('这次没真正调用 embedding，原因是：没有 API key，也没有开启默认凭证。')
    elif 使用默认凭证:
        print('你开启了默认凭证，但 embedding client 仍未建立。通常要检查是否已 az login，或当前凭证是否可用。')
    else:
        print('你提供了 key 或相关配置，但 embedding client 仍未建立。建议检查 key 是否有效、部署名是否正确。')
    print()

    print('三、下一步建议')
    if binder_obj.embedding_client is None:
        print('方案1：在 notebook 里直接设置 API_KEY = 你的有效 key')
        print("方案2：先执行 os.environ['AZURE_AI_API_KEY'] = '你的有效 key'")
        print('方案3：如果你本机已经 az login，就把 使用默认凭证 = True')
    else:
        print('当前无需再补环境，直接运行后面的绑定单元即可。')


中文Embedding自检(
    binder,
    启用Embedding=启用Embedding,
    使用默认凭证=使用默认凭证,
    API_KEY=API_KEY,
)

一、当前状态
启用Embedding开关： True
已加载Embedding索引： True
本次可调用Embedding： True
环境里有Key： False
手动传入API_KEY： True
使用默认凭证： False

二、诊断结论
本次已经具备调用 embedding 的能力。只要后续 bind 正常执行，就会真的发起 embedding 检索。

三、下一步建议
当前无需再补环境，直接运行后面的绑定单元即可。


## 5. 执行绑定

In [22]:
result = binder.bind(
    QUESTION_RECORD,
    top_k=TOP_K,
    max_secondary=MAX_SECONDARY,
    coarse_k=COARSE_K,
    global_recall=GLOBAL_RECALL,
    sparse_recall=SPARSE_RECALL,
    embedding_recall=EMBEDDING_RECALL,
)

binder.write_output(result, OUTPUT_JSON)

print('绑定已完成。')
print('结果文件：', OUTPUT_JSON)
print(json.dumps(summarize_result(result), ensure_ascii=False, indent=2))

绑定已完成。
结果文件： /Users/xumuchi/Desktop/TeachAgent/scratch/wrong_question_binder_playground/binder_test_output.json
{
  "question_id": "wq_0c00e962c5a7",
  "primary_node_id": "math.解析几何.直线的方程.直线倾斜角与斜率",
  "secondary_node_ids": [],
  "binding_confidence": 0.3174,
  "top_k_node_ids": [
    "math.解析几何.直线的方程.直线倾斜角与斜率",
    "math.解析几何.直线的方程.直线斜截式方程",
    "math.解析几何.圆锥曲线.双曲线离心率与渐近线",
    "math.解析几何.直线的方程.直线点斜式方程",
    "math.解析几何.直线的方程.直线两点式方程"
  ],
  "coarse_subtrees": [
    "math.集合_映射_函数与导数.导数",
    "math.解析几何.直线的方程",
    "math.集合_映射_函数与导数.函数.基本初等函数"
  ],
  "candidate_pool_size": 53
}


## 6. 中文说明版结果

这一格会把结果翻译成更容易看的中文说明。

In [23]:
def 读取备注值(备注列表, 前缀):
    for item in 备注列表:
        if item.startswith(前缀):
            return item[len(前缀):]
    return None


def 中文展示(result_obj):
    record = result_obj.binding_record
    primary = record['primary_binding']
    notes = record['binding_notes']

    print('一、主绑定结果')
    print('主绑定知识点：', primary['title'])
    print('node_id：', primary['node_id'])
    print('绑定置信度：', record['binding_confidence'])
    print('主绑定理由：', primary['reason'])
    print()

    print('二、这次是否真的用了 embedding')
    print('embedding_recall_enabled =', 读取备注值(notes, 'embedding_recall_enabled='))
    print('embedding_candidate_count =', 读取备注值(notes, 'embedding_candidate_count='))
    print('sparse_candidate_count =', 读取备注值(notes, 'sparse_candidate_count='))
    print()

    print('三、粗路由结果（先判断大概属于哪个分支）')
    for item in result_obj.coarse_subtree_candidates:
        print(f"- {item['node_id']} | 分数={item['score']} | 子叶子数={item['child_leaf_count']}")
    print()

    print('四、Top-K 候选（最终排序）')
    for entry in record['top_k_candidates']:
        breakdown = entry['score_breakdown']
        print(f"[{entry['rank']}] {entry['title']}")
        print('  node_id：', entry['node_id'])
        print('  总分：', entry['bind_score'])
        print(
            '  分数拆解：'
            f"语义={breakdown['semantic_match_score']} | "
            f"关键词={breakdown['keyword_match_score']} | "
            f"解析贴合={breakdown['solution_alignment_score']} | "
            f"粒度={breakdown['granularity_fit_score']} | "
            f"聚焦={breakdown.get('focus_alignment_score', 0.0)} | "
            f"稀疏={breakdown['bm25_score']} | "
            f"向量={breakdown['embedding_score']} | "
            f"规则={breakdown['rule_score']}"
        )
        print('  解释：', entry['reason'])
        print()


中文展示(result)

一、主绑定结果
主绑定知识点： 直线倾斜角与斜率
node_id： math.解析几何.直线的方程.直线倾斜角与斜率
绑定置信度： 0.3174
主绑定理由： 命中关键词：斜率、直线方程；解析模式匹配：直线方程；该叶子对题目核心求解动作的解释最完整

二、这次是否真的用了 embedding
embedding_recall_enabled = true
embedding_candidate_count = 24
sparse_candidate_count = 24

三、粗路由结果（先判断大概属于哪个分支）
- math.集合_映射_函数与导数.导数 | 分数=0.2762 | 子叶子数=9
- math.解析几何.直线的方程 | 分数=0.1458 | 子叶子数=10
- math.集合_映射_函数与导数.函数.基本初等函数 | 分数=0.1352 | 子叶子数=6

四、Top-K 候选（最终排序）
[1] 直线倾斜角与斜率
  node_id： math.解析几何.直线的方程.直线倾斜角与斜率
  总分： 0.4091
  分数拆解：语义=0.1061 | 关键词=0.2569 | 解析贴合=0.1436 | 粒度=0.584 | 稀疏=1.0 | 向量=0.8011 | 规则=0.2856
  解释： 命中关键词：斜率、直线方程；解析模式匹配：直线方程；该叶子对题目核心求解动作的解释最完整

[2] 直线斜截式方程
  node_id： math.解析几何.直线的方程.直线斜截式方程
  总分： 0.3887
  分数拆解：语义=0.1184 | 关键词=0.1338 | 解析贴合=0.1729 | 粒度=0.5308 | 稀疏=1.0 | 向量=0.8262 | 规则=0.1756
  解释： 命中关键词：斜率；解析模式匹配：标准答案与该公式卡的表达式或使用条件接近

[3] 双曲线离心率与渐近线
  node_id： math.解析几何.圆锥曲线.双曲线离心率与渐近线
  总分： 0.3793
  分数拆解：语义=0.1328 | 关键词=0.1093 | 解析贴合=0.1673 | 粒度=0.506 | 稀疏=1.0 | 向量=0.8194 | 规则=0.11
  解释： 命中关键词：渐近线；解析模式匹配：渐近线

[4] 直线点斜式方程
  n

## 7. 精简查看版

如果你只想快速看主绑定、备注和置信度，可以看这一格。

In [8]:
record = result.binding_record
精简视图 = {
    '主绑定': record['primary_binding'],
    '辅助绑定': record['secondary_bindings'],
    '绑定置信度': record['binding_confidence'],
    '运行备注': record['binding_notes'],
}

print(json.dumps(精简视图, ensure_ascii=False, indent=2))

{
  "主绑定": {
    "node_id": "math.解析几何.直线的方程.直线倾斜角与斜率",
    "rank": 1,
    "bind_score": 0.329,
    "title": "直线倾斜角与斜率",
    "node_kind": "formula",
    "review_role": "core",
    "binding_role": "primary_allowed",
    "path": [
      "数学",
      "解析几何",
      "直线的方程",
      "直线倾斜角与斜率"
    ],
    "score_breakdown": {
      "semantic_match_score": 0.1061,
      "keyword_match_score": 0.2569,
      "solution_alignment_score": 0.1436,
      "granularity_fit_score": 0.584,
      "binding_role_score": 1.0,
      "bm25_score": 1.0,
      "embedding_score": 0.0,
      "rule_score": 0.2856
    },
    "evidence": {
      "matched_keywords": [
        "斜率",
        "直线方程"
      ],
      "matched_aliases": [],
      "matched_path_terms": [],
      "matched_solution_pattern": "直线方程",
      "matched_error_pattern": null,
      "quoted_fragments": [
        "...l交C于P，Q两点，直线AP，AQ的斜率之和为0．求l的斜率",
        "...2/2)x.  （2）求直线 l 的斜率\n设直线 l 的方程为  y = k...",
        "...\n  ( AP )、( AQ ) 的斜率之和为 0，所以 ( \frac{

## 8. 可选：和“关闭 embedding”对比

如果你想测试它的通用性，可以跑这一格，直接比较有无 embedding 的差别。

In [9]:
binder_no_embedding = WrongQuestionBinder.from_environment(enable_embeddings=False)
result_no_embedding = binder_no_embedding.bind(
    QUESTION_RECORD,
    top_k=TOP_K,
    max_secondary=MAX_SECONDARY,
    coarse_k=COARSE_K,
    global_recall=GLOBAL_RECALL,
    sparse_recall=SPARSE_RECALL,
    embedding_recall=EMBEDDING_RECALL,
)
binder_no_embedding.write_output(result_no_embedding, OUTPUT_NO_EMBED_JSON)

对比结果 = {
    '开启Embedding': summarize_result(result),
    '关闭Embedding': summarize_result(result_no_embedding),
}

print(json.dumps(对比结果, ensure_ascii=False, indent=2))

{
  "开启Embedding": {
    "question_id": "wq_0c00e962c5a7",
    "primary_node_id": "math.解析几何.直线的方程.直线倾斜角与斜率",
    "secondary_node_ids": [],
    "binding_confidence": 0.2625,
    "top_k_node_ids": [
      "math.解析几何.直线的方程.直线倾斜角与斜率",
      "math.解析几何.直线的方程.直线斜截式方程",
      "math.解析几何.圆锥曲线.双曲线离心率与渐近线",
      "math.解析几何.直线的方程.直线点斜式方程",
      "math.解析几何.直线的方程.直线两点式方程"
    ],
    "coarse_subtrees": [
      "math.集合_映射_函数与导数.导数",
      "math.解析几何.直线的方程",
      "math.集合_映射_函数与导数.函数.基本初等函数"
    ],
    "candidate_pool_size": 46
  },
  "关闭Embedding": {
    "question_id": "wq_0c00e962c5a7",
    "primary_node_id": "math.解析几何.直线的方程.直线倾斜角与斜率",
    "secondary_node_ids": [],
    "binding_confidence": 0.2625,
    "top_k_node_ids": [
      "math.解析几何.直线的方程.直线倾斜角与斜率",
      "math.解析几何.直线的方程.直线斜截式方程",
      "math.解析几何.圆锥曲线.双曲线离心率与渐近线",
      "math.解析几何.直线的方程.直线点斜式方程",
      "math.解析几何.直线的方程.直线两点式方程"
    ],
    "coarse_subtrees": [
      "math.集合_映射_函数与导数.导数",
      "math.解析几何.直线的方程",
      "math.集合_映射_函数与导数

## 9. 查看完整 JSON 输出

如果你之后想把完整输出发给我，就直接发这个文件内容。

In [20]:
print(OUTPUT_JSON)
print(OUTPUT_JSON.read_text(encoding='utf-8')[:5000])

/Users/xumuchi/Desktop/TeachAgent/scratch/wrong_question_binder_playground/binder_test_output.json
{
  "question_id": "wq_584afe162a42",
  "question_payload": {
    "stem": "已知数列 a_n 满足 a_{n+1}=3a_n+1，且 a_1=1/2，求证：数列 {a_n+1/2} 为等比数列。",
    "question_type": "证明题",
    "correct_answer": "数列 {a_n+1/2} 是以 1 为首项、3 为公比的等比数列。",
    "solution_text": "由 a_{n+1}=3a_n+1，得 a_{n+1}+1/2=3a_n+1+1/2=3(a_n+1/2)。\n又 a_1+1/2=1/2+1/2=1，因此新数列满足后一项等于前一项乘 3，且首项为 1，\n所以 {a_n+1/2} 是首项为 1、公比为 3 的等比数列。",
    "source_name": "binder_notebook_demo",
    "source_section": "请按需要修改",
    "tags": [
      "数列",
      "递推",
      "等比数列"
    ]
  },
  "primary_binding": {
    "node_id": "math.数列与不等式.数列.等差_等比数列.等比数列通项公式",
    "rank": 1,
    "bind_score": 0.3448,
    "title": "等比数列通项公式",
    "node_kind": "formula",
    "review_role": "core",
    "binding_role": "primary_allowed",
    "path": [
      "数学",
      "数列与不等式",
      "数列",
      "等差、等比数列",
      "等比数列通项公式"
    ],
    "score_breakdown": {
      "semantic_match_score